# 第90章 逻辑回归分类

<!-- module-learning-arc:start -->
> **机器学习 模块主线｜第 5 / 34 步：扩展监督/无监督模型工具箱**
>
> **持续应用背景：** 建设可信预测系统：从统一训练流程开始，比较模型、处理不平衡、选择阈值、解释结果并保存完整 Pipeline，最终回答模型能否安全投入使用。
>
> **承接上一阶段：** 线性回归与正则化  →  **本章任务：** 逻辑回归分类  →  **下一步：** K近邻模型（KNN）
>
> **大作业连接：** 本章练习将成为《模型上线评审会》的一部分，最终需要把候选模型变成经过预测合同、泄漏审计、业务阈值、错误分析和模型卡检查的上线建议。
<!-- module-learning-arc:end -->


## 本章场景

**背景引入**：很多实际的业务问题都可以归结成一个「是或否」的判断——肿瘤是良性还是恶性、用户会不会流失、一封邮件是不是垃圾。逻辑回归正是这类分类任务最常用的起点，它不只告诉你结果，还会输出一个介于 0 到 1 之间的概率，让你同时看到模型是否确信。


## 本章目标

学完本章，你将能够：

- **理解**：理解「逻辑回归分类」的核心思想、适用场景、关键假设与要解释的业务问题。
- **操作**：能按标准流程完成数据准备、模型训练与评估，并解读「逻辑回归分类」的关键输出指标。
- **迁移**：能把「逻辑回归分类」迁移到一份新数据上，独立完成任务并就结果给出有分寸的结论。


## 核心概念

**背景引入**：很多问题不是“预测多少钱”，而是“判断会不会”——比如这个客户会不会下单、这条消息是不是垃圾。逻辑回归给的不是 0/1 的硬判决，而是“属于正类的概率”。它把概率与几率的对数挂钩，再用一个默认阈值（通常是 0.5）把概率切回类别。看懂它，你才能理解为什么阈值和系数解释那么关键。


- 逻辑回归建模的是对数胜算
- 默认阈值通常为 0.5（打个比方：它给的是“下雨概率70%”这种把握度，不是直接喊“明天下雨”；要不要带伞，得你按成本自己定个线。）
- ROC-AUC 衡量随机正例排在随机负例之前的概率
- 系数解释依赖特征尺度与共线性


## 方法分类速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 乳腺癌二分类 | `.fit()` | 使用标准化流水线，避免特征量纲影响优化与正则化。 | 把 predict 输出当成概率 |
| 概率、阈值与指标 | `model.predict_proba()`、`.astype()`、`.tolist()` | 改变阈值会在精确率与召回率之间移动。 | 只按准确率选择阈值 |


## 例 1｜乳腺癌二分类

使用标准化流水线，避免特征量纲影响优化与正则化。


<!-- math-foundation:chapter-90 -->
### 数学推导｜逻辑回归把线性得分映射为概率

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜线性部分给出任意实数得分。** $z=\beta_0+x^T\beta$。

**第 2 步｜把概率的赔率写成线性。** 逻辑回归假设

$$
\log\frac{p}{1-p}=z
$$

两边取指数并解出 $p$，得到 $p=1/(1+e^{-z})$。

**第 3 步｜用概率损失拟合。** 单个二分类样本的负对数似然为

$$
\ell_i=-\bigl[y_i\log p_i+(1-y_i)\log(1-p_i)\bigr]
$$

训练优化概率质量，部署阈值则由错误成本另行决定。

**把上面的关系收束为本章计算式：**

$$
p(y=1\mid x)=\sigma(z)=\frac{1}{1+e^{-z}},\qquad z=\beta_0+x^T\beta
$$

**符号解释：** $p$ 是正类概率，分类阈值不必固定为 0.5。

**代码对应：** 使用 `predict_proba` 得到概率，再按业务成本选择阈值。

**使用边界：** 未校准的概率和相关系数不能直接解释为真实因果影响。


In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, random_state=79
)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(
    X_train, y_train
)


In [ ]:
# （自动维护）练习上下文快照 1：参考答案将基于此刻的变量运行
_pds_snap_1 = dict(globals())


**练一练**：试着只改一个参数——把上面训练单元里模型的正则化参数 `C` 从默认值改成 `0.01`，重新训练得到 `model2`，然后核对它的 `predict_proba` 是否仍是合法概率（每个样本的类别概率加和等于 1）。改小 `C` 会增强正则化、把系数压得更靠近 0，模型的分类能力可能略微下降，但输出的概率结构应当保持不变。先在上面的示例单元复制并修改训练代码，再运行下面的自检单元格。


In [ ]:
try:
    pass
    # 请在下方填写代码
    # 练一练：把示例模型的 C 改成 0.01，重新训练并存入 model2。
    # 提示：复制上方示例的 make_pipeline 训练代码，把 LogisticRegression(max_iter=1000) 改成
    # LogisticRegression(C=0.01, max_iter=1000)。

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 例 2｜概率、阈值与指标

改变阈值会在精确率与召回率之间移动。


In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    confusion_matrix,
)

prob = model.predict_proba(X_test)[:, 1]
print("ROC-AUC:", round(roc_auc_score(y_test, prob), 3))
for threshold in [0.3, 0.5, 0.7]:
    pred = (prob >= threshold).astype(int)
    print(
        threshold,
        "precision=",
        round(precision_score(y_test, pred), 3),
        "recall=",
        round(recall_score(y_test, pred), 3),
        "matrix=",
        confusion_matrix(y_test, pred).tolist(),
    )


## 独立迁移练习

在不改变数据切分和指标的前提下，比较基线与一个模型设置。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# （自动维护）练习上下文快照 2：参考答案将基于此刻的变量运行
_pds_snap_2 = dict(globals())


In [ ]:
try:
    # TODO: 在此粘贴或改写最接近的示例。
    # 记录：我改了什么？预期会发生什么？实际观察到什么？
    change_note = "待填写"
    expected_change = "待填写"
    observed_change = "运行后填写"
    print(
        {"修改": change_note, "预期": expected_change, "观察": observed_change}
    )

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 本章实训：模型与基线比较

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

X = pd.DataFrame(
    {"visits": [1, 2, 3, 4, 5, 6], "discount": [0, 0, 1, 1, 1, 2]}
)
y = np.array([12, 15, 19, 23, 27, 31])
baseline = DummyRegressor(strategy="mean").fit(X, y)
_demo_model = LinearRegression().fit(X, y)
print("基线预测：", np.round(baseline.predict(X[:2]), 2))
print("模型预测：", np.round(_demo_model.predict(X[:2]), 2))
print("基线MAE：", round(mean_absolute_error(y, baseline.predict(X)), 2))
print("模型MAE：", round(mean_absolute_error(y, _demo_model.predict(X)), 2))


### 第一个结果怎么读

复杂模型之前先建立基线。只有在同一数据切分和同一指标下超过基线，模型才值得继续分析。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
X_changed = X.copy()
X_changed["visits"] = X_changed["visits"] + 1
changed_prediction = _demo_model.predict(X_changed)
print("原始前2个预测：", np.round(_demo_model.predict(X[:2]), 2))
print("访问次数+1后的预测：", np.round(changed_prediction[:2], 2))
print(
    "预测变化：",
    np.round(changed_prediction[:2] - _demo_model.predict(X[:2]), 2),
)


### 第二个结果怎么读

只把一个特征整体加 1，观察预测变化。这个实验只能说明模型的预测响应，不能直接证明真实世界的因果关系。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：模型特征泄漏怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd

_demo_data = pd.DataFrame(
    {
        "visits": [2, 4, 6],
        "duration_after_call": [30, 80, 120],
        "target": [0, 1, 1],
    }
)
forbidden = {"target", "duration_after_call"}
features = [column for column in _demo_data.columns if column not in forbidden]
print("禁止使用：", sorted(forbidden))
print("安全特征：", features)
print("原因：特征必须在预测时点已经可获得。")


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

如果一个字段在结果发生之后才产生，它即使与目标高度相关，也不能作为预测特征。先定义预测时点，再列可用字段。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 易错点提醒

- 把 predict 输出当成概率
- 只按准确率选择阈值
- 忘记确认哪个标签是正类
- 将系数大小直接当作变量重要性或因果效应


## 练习与作业

1. 寻找召回率至少 0.95 的最高阈值
2. 报告该阈值下精确率
3. 输出混淆矩阵

提交前检查：代码可从上到下运行，关键中间结果可核对，结论注明计算口径。

## 90.11 练习路径

1. **跟练**：先运行示例，确认输出结构，再完成“寻找召回率至少 0.95 的最高阈值”。
2. **独立完成**：不复制示例代码，完成“报告该阈值下精确率”，并保留一个中间结果用于检查。
3. **迁移挑战**：尝试“输出混淆矩阵”，用一两句话说明你修改了什么。

### 90.11.1 完成标准

- 代码从上到下运行不报错，关键变量类型和形状符合预期。
- 至少输出一个可核对的数值、表格或图形，并写明计算口径。
- 结论能够回答任务问题，同时说明一个限制或未验证的假设。

### 90.11.2 分级提示

- **提示 1**：先复用示例中的数据结构和变量命名。
- **提示 2**：把任务拆成“准备数据 → 计算 → 检查 → 表达”四步。
- **提示 3**：运行隐藏答案前，先用 type()、shape、head() 或断言定位问题。


## 小结

使用逻辑回归输出分类概率，理解系数、决策阈值、混淆矩阵和 ROC-AUC。


### 你已经掌握

- 训练二分类逻辑回归
- 使用 predict_proba 获取概率
- 区分概率排序与阈值分类
- 解释标准化后的系数方向


### 需要注意

- 把 predict 输出当成概率
- 只按准确率选择阈值
- 忘记确认哪个标签是正类
- 将系数大小直接当作变量重要性或因果效应


## 参考答案


### 本章练习


In [ ]:
# 恢复练习 1 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_1)


In [ ]:
# 参考答案：改小正则化参数 C 重新训练
model2 = make_pipeline(
    StandardScaler(), LogisticRegression(C=0.01, max_iter=1000)
).fit(X_train, y_train)


### 本章练习


In [ ]:
# 恢复练习 2 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_2)


In [ ]:
candidates = []
for threshold in [i / 100 for i in range(5, 96)]:
    p = (prob >= threshold).astype(int)
    r = recall_score(y_test, p)
    if r >= 0.95:
        candidates.append((threshold, precision_score(y_test, p), r))
best_threshold, best_precision, best_recall = max(
    candidates, key=lambda x: x[0]
)
print(best_threshold, round(best_precision, 3), round(best_recall, 3))
